# 1. Data Cleaning

Loads the raw California Housing dataset and applies cleaning steps:
- Handles missing values in `total_bedrooms` (median imputation, grouped by `ocean_proximity`)
- Removes duplicate rows
- Flags/optionally drops rows where `median_house_value` is capped at 500001
- Basic sanity checks on value ranges

This notebook automatically finds the project root, so it works whether you launch Jupyter from the project root, `notebooks/`, or `notebooks/pipeline/`.

In [1]:
from pathlib import Path

def find_project_root(marker="requirements.txt", max_search_depth=4):
    """Locates the housing_project root so paths work no matter where
    Jupyter was launched from. Two strategies, tried in order:
    1. Walk UPWARD from the current directory (covers launching Jupyter
       from inside the project, e.g. from notebooks/pipeline/).
    2. Search DOWNWARD into subfolders (covers the common case of
       launching Jupyter from your home folder or Desktop, then browsing
       into the project through the Jupyter file browser -- the kernel's
       working directory stays at the launch folder, not the notebook's).
    """
    start = Path.cwd().resolve()

    # Strategy 1: search upward
    for parent in [start] + list(start.parents):
        if (parent / marker).exists():
            return parent

    # Strategy 2: search downward (breadth-first, limited depth)
    frontier = [start]
    for _ in range(max_search_depth):
        next_frontier = []
        for folder in frontier:
            try:
                subdirs = [d for d in folder.iterdir() if d.is_dir() and not d.name.startswith(".")]
            except PermissionError:
                continue
            for d in subdirs:
                if (d / marker).exists():
                    return d
                next_frontier.append(d)
        frontier = next_frontier
        if not frontier:
            break

    raise FileNotFoundError(
        f"Could not locate the project root (looking for '{marker}') starting from {start}.\n"
        f"Fix: either launch Jupyter from inside the housing_project folder, "
        f"or set PROJECT_ROOT manually below, e.g.:\n"
        f'    PROJECT_ROOT = Path(r"C:\\path\\to\\housing_project")'
    )

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

Project root: D:\Projectsinterns\housing_project


In [2]:
import pandas as pd
import numpy as np

RAW_PATH = PROJECT_ROOT / "data" / "housing.csv"
CLEAN_OUT_PATH = PROJECT_ROOT / "data" / "housing_clean.csv" 

## Load raw data

In [3]:
def load_raw_data(path=RAW_PATH):
    df = pd.read_csv(path)
    return df

df = load_raw_data()
print(df.shape)
df.head()

(20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


## Cleaning function

Encapsulates all cleaning steps so it can be imported and reused by `feature_engineering` or called directly here.

In [4]:
def clean_data(df, drop_capped_target=False, verbose=True):
    df = df.copy()

    # 1. Duplicates
    n_dupes = df.duplicated().sum()
    if n_dupes > 0:
        df = df.drop_duplicates()
    if verbose:
        print(f"Dropped {n_dupes} duplicate rows.")

    # 2. Missing values in total_bedrooms -> median imputation grouped by ocean_proximity
    n_missing = df["total_bedrooms"].isnull().sum()
    df["total_bedrooms"] = df.groupby("ocean_proximity")["total_bedrooms"].transform(
        lambda x: x.fillna(x.median())
    )
    if verbose:
        print(f"Imputed {n_missing} missing values in total_bedrooms (median per ocean_proximity group).")

    # 3. Sanity checks: California lat/long bounds
    lat_ok = df["latitude"].between(32, 43).all()
    lon_ok = df["longitude"].between(-125, -113).all()
    if verbose:
        print(f"Latitude within CA bounds: {lat_ok} | Longitude within CA bounds: {lon_ok}")

    # 4. No negative values in count-based columns
    count_cols = ["total_rooms", "total_bedrooms", "population", "households"]
    for col in count_cols:
        n_negative = (df[col] < 0).sum()
        if n_negative > 0 and verbose:
            print(f"WARNING: {n_negative} negative values found in {col}")

    # 5. Flag capped target values (known artifact of this dataset: max value is 500001)
    n_capped = (df["median_house_value"] >= 500001).sum()
    if verbose:
        print(f"Rows with median_house_value at/above the 500001 cap: {n_capped} "
              f"({n_capped / len(df) * 100:.2f}% of data)")

    if drop_capped_target:
        df = df[df["median_house_value"] < 500001]
        if verbose:
            print("Dropped capped target rows (drop_capped_target=True).")

    df = df.reset_index(drop=True)
    return df

## Run cleaning and save output

In [5]:
df_clean = clean_data(df)
df_clean.to_csv(CLEAN_OUT_PATH, index=False)
print(f"\nSaved cleaned dataset -> {CLEAN_OUT_PATH}  (shape: {df_clean.shape})")
df_clean.head()

Dropped 0 duplicate rows.
Imputed 207 missing values in total_bedrooms (median per ocean_proximity group).
Latitude within CA bounds: True | Longitude within CA bounds: True
Rows with median_house_value at/above the 500001 cap: 965 (4.68% of data)

Saved cleaned dataset -> D:\Projectsinterns\housing_project\data\housing_clean.csv  (shape: (20640, 10))


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY
